In [4]:
"""
main_allocations.py
====================
Compute feature allocations across 6 datasets x 3 models x 5 GT methods.

Key design notes:
  - Datasets are pre-split: X_train/X_test/y_train/y_test loaded directly
  - UtilityWrapper.data is a DataObject (not a list) because gt_concepts
    functions call both len(utility.data) AND utility.data.data_names
  - get_model called with random_state=42 for reproducibility
  - Per-method try/except: one failure does not abort the whole combo
  - Lambda default-arg trick avoids late-binding bug in method dispatch loop
  - Results saved as CSV: rows=features, cols=methods
  - Diagnostic values (v_N, singletons, LOO) saved in a separate CSV
"""

import csv
import os
import pickle

import numpy as np
from sklearn.metrics import accuracy_score

from cgt_xai.models import get_model

from cgt_xai.concepts.shapley import exact_shapley_value
from cgt_xai.concepts.least_core import exact_least_core
from cgt_xai.concepts.nucleolus import exact_nucleolus
from cgt_xai.concepts.gately_point import exact_gately_value
from cgt_xai.concepts.banzhaf import exact_banzhaf_index

# Config
DATASETS = [
    "../../datasets/wine_preprocessed_core.npz",
    "../../datasets/yield_preprocessed.npz",
    "../../datasets/wine_preprocessed.npz",
    "../../datasets/soil_spi_preprocessed.npz",
    "../../datasets/house_preprocessed.npz",
    "../../datasets/bcw_preprocessed.npz",
]

MODELS        = ["logistic", "svm", "random_forest"]
METHODS       = ["shapley", "banzhaf", "gately", "least_core", "nucleolus"]
MAX_FEATURES  = 20
RANDOM_STATE  = 42
OUTPUT_DIR    = "./output_final_allocations"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# DataObject
class DataObject:
    def __init__(self, n: int, feature_names: list):
        self.data_names = feature_names
        self._n = n

    def __len__(self):
        return self._n


# AccuracyUtility
class AccuracyUtility:
    def __init__(self, X_train, X_test, y_train, y_test, model_name,
                 random_state=RANDOM_STATE):
        self.X_train      = X_train
        self.X_test       = X_test
        self.y_train      = y_train
        self.y_test       = y_test
        self.model_name   = model_name
        self.random_state = random_state
        self.n            = X_train.shape[1]
        self.cache        = {}

    def __call__(self, S):
        key = tuple(sorted(int(i) for i in S))

        if key in self.cache:
            return self.cache[key]

        if len(key) == 0:
            self.cache[key] = 0.0
            return 0.0

        X_tr = self.X_train[:, key]
        X_te = self.X_test[:, key]

        model = get_model(self.model_name, random_state=self.random_state)

        try:
            model.fit(X_tr, self.y_train)
            acc = accuracy_score(self.y_test, model.predict(X_te))
        except Exception as e:
            print(f"  [Utility Error] model={self.model_name}, S={key}, error={e}")
            acc = 0.0

        self.cache[key] = acc
        return acc


# UtilityWrapper
class UtilityWrapper:
    def __init__(self, utility: AccuracyUtility, feature_names: list):
        self.util = utility
        self.data = DataObject(utility.n, feature_names)

    def __call__(self, S):
        return self.util(S)


# CSV save helpers
def save_allocations_csv(save_path, feature_names, results, methods):
    """
    Rows = features, Columns = methods.

    feature_name | shapley | banzhaf | gately | least_core | nucleolus
    """
    with open(save_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["feature_name"] + methods)
        for i, fname in enumerate(feature_names):
            row = [fname]
            for m in methods:
                val = results.get(m)
                row.append(round(float(val[i]), 8) if val is not None else "ERROR")
            writer.writerow(row)
    print(f"    Allocations --> {save_path}")


def save_diagnostics_csv(save_path, feature_names, utility, n_features):
    """
    Diagnostic utility values per feature:
      feature_name | v_singleton | v_N_minus_i | marginal_to_grand
    Plus a summary row for v_N and sum_singletons.
    """
    N = list(range(n_features))
    v_N = utility(N)
    with open(save_path, "w", newline="") as f:
        writer = csv.writer(f)
        # summary rows at top
        writer.writerow(["metric", "value"])
        writer.writerow(["v_N (grand coalition accuracy)", round(v_N, 8)])
        writer.writerow(["sum_v_singletons", round(sum(utility([i]) for i in N), 8)])
        writer.writerow([])
        # per-feature rows
        writer.writerow(["feature_name", "v_singleton", "v_N_minus_i", "marginal_to_grand"])
        for i, fname in enumerate(feature_names):
            v_s   = utility([i])
            v_loo = utility([j for j in N if j != i])
            marginal = v_N - v_loo
            writer.writerow([fname, round(v_s, 8), round(v_loo, 8), round(marginal, 8)])
    print(f"    Diagnostics --> {save_path}")


# Main loop
for dataset_path in DATASETS:
    print(f"\n{'='*60}")
    print(f"DATASET: {dataset_path}")
    print(f"{'='*60}")

    data = np.load(dataset_path, allow_pickle=True)
    available_keys = list(data.keys())
    print(f"  Keys in archive: {available_keys}")

    required = ["X_train", "X_test", "y_train", "y_test"]
    if not all(k in available_keys for k in required):
        print(f"  [SKIP] Missing required keys. Expected {required}, found {available_keys}")
        continue

    X_train = data["X_train"]
    X_test  = data["X_test"]
    y_train = data["y_train"]
    y_test  = data["y_test"]
    feature_names = (data["feature_names"].tolist()
                     if "feature_names" in available_keys
                     else [f"f{i}" for i in range(X_train.shape[1])])

    n_features = X_train.shape[1]
    print(f"  X_train={X_train.shape}  X_test={X_test.shape}")
    print(f"  features={feature_names}")

    if n_features > MAX_FEATURES:
        print(f"  [SKIP] {n_features} features > MAX_FEATURES={MAX_FEATURES}")
        continue

    ds_name = os.path.basename(dataset_path).replace(".npz", "")

    for model_name in MODELS:
        print(f"\n  --- Model: {model_name} ---")

        utility = AccuracyUtility(X_train, X_test, y_train, y_test, model_name)
        u = UtilityWrapper(utility, feature_names)

        results = {}

        for method_name, compute_fn in [
            ("shapley",    lambda fn=exact_shapley_value: fn(u)),
            ("banzhaf",    lambda fn=exact_banzhaf_index: fn(u)),
            ("gately",     lambda fn=exact_gately_value:  fn(u)),
            ("least_core", lambda fn=exact_least_core:    fn(u)),
            ("nucleolus",  lambda fn=exact_nucleolus:     fn(u)),
        ]:
            print(f"    Computing {method_name} ...", end=" ", flush=True)
            try:
                result = compute_fn()
                results[method_name] = result.values if hasattr(result, "values") else result
                print("done")
            except Exception as e:
                print(f"ERROR: {e}")
                results[method_name] = None

        alloc_path = os.path.join(OUTPUT_DIR, f"{ds_name}__{model_name}__allocations.csv")
        save_allocations_csv(alloc_path, feature_names, results, METHODS)

        diag_path = os.path.join(OUTPUT_DIR, f"{ds_name}__{model_name}__diagnostics.csv")
        save_diagnostics_csv(diag_path, feature_names, utility, n_features)

print("\nAll done.")


DATASET: ../datasets/wine_preprocessed_core.npz
  Keys in archive: ['X_train', 'X_test', 'y_train', 'y_test', 'feature_names']
  X_train=(142, 13)  X_test=(36, 13)
  features=['Alcohol', 'Malic_acid', 'Ash', 'Alcalinity_of_ash', 'Magnesium', 'Total_phenols', 'Flavanoids', 'Nonflavanoid_phenols', 'Proanthocyanins', 'Color_intensity', 'Hue', 'OD280_OD315', 'Proline']

  --- Model: logistic ---
    Computing shapley ... done
    Computing banzhaf ... done
    Computing gately ... done
    Computing least_core ... done
    Computing nucleolus ... done
    Allocations --> ./output_final_allocations/wine_preprocessed_core__logistic__allocations.csv
    Diagnostics --> ./output_final_allocations/wine_preprocessed_core__logistic__diagnostics.csv

  --- Model: svm ---
    Computing shapley ... done
    Computing banzhaf ... done
    Computing gately ... done
    Computing least_core ... done
    Computing nucleolus ... done
    Allocations --> ./output_final_allocations/wine_preprocessed_core_

In [10]:
"""
main_allocations.py
====================
Compute feature allocations across 6 datasets x 3 models x 5 GT methods.

Key design notes:
  - Datasets are pre-split: X_train/X_test/y_train/y_test loaded directly
  - UtilityWrapper.data is a DataObject (not a list) because gt_concepts
    functions call both len(utility.data) AND utility.data.data_names
  - get_model called with random_state=42 for reproducibility
  - Per-method try/except: one failure does not abort the whole combo
  - Lambda default-arg trick avoids late-binding bug in method dispatch loop
  - Results saved as CSV: rows=features, cols=methods
  - Diagnostic values (v_N, singletons, LOO) saved in a separate CSV
"""

import csv
import os

import numpy as np
from sklearn.metrics import accuracy_score

from cgt_xai.models import get_model

from cgt_xai.concepts.shapley import exact_shapley_value
from cgt_xai.concepts.least_core import exact_least_core
from cgt_xai.concepts.gately_point import exact_gately_value
from cgt_xai.concepts.banzhaf import exact_banzhaf_index

# Config
DATASETS = [
    "../../datasets/house_preprocessed.npz",
    "../../datasets/bcw_preprocessed.npz",
]

MODELS               = ["logistic", "svm", "random_forest"]
MAX_FEATURES         = 20
METHODS              = ["shapley", "banzhaf", "gately", "least_core"]
RANDOM_STATE         = 42
OUTPUT_DIR           = "./output_final_allocations"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# DataObject
class DataObject:
    def __init__(self, n: int, feature_names: list):
        self.data_names = feature_names
        self._n = n

    def __len__(self):
        return self._n


# AccuracyUtility
class AccuracyUtility:
    def __init__(self, X_train, X_test, y_train, y_test, model_name,
                 random_state=RANDOM_STATE):
        self.X_train      = X_train
        self.X_test       = X_test
        self.y_train      = y_train
        self.y_test       = y_test
        self.model_name   = model_name
        self.random_state = random_state
        self.n            = X_train.shape[1]
        self.cache        = {}

    def __call__(self, S):
        key = tuple(sorted(int(i) for i in S))

        if key in self.cache:
            return self.cache[key]

        if len(key) == 0:
            self.cache[key] = 0.0
            return 0.0

        X_tr = self.X_train[:, key]
        X_te = self.X_test[:, key]

        model = get_model(self.model_name, random_state=self.random_state)

        try:
            model.fit(X_tr, self.y_train)
            acc = accuracy_score(self.y_test, model.predict(X_te))
        except Exception as e:
            print(f"  [Utility Error] model={self.model_name}, S={key}, error={e}")
            acc = 0.0

        self.cache[key] = acc
        return acc


# UtilityWrapper
class UtilityWrapper:
    def __init__(self, utility: AccuracyUtility, feature_names: list):
        self.util = utility
        self.data = DataObject(utility.n, feature_names)

    def __call__(self, S):
        return self.util(S)


# CSV save helpers
def save_allocations_csv(save_path, feature_names, results, methods):
    """
    Rows = features, Columns = methods.
    """
    with open(save_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["feature_name"] + methods)
        for i, fname in enumerate(feature_names):
            row = [fname]
            for m in methods:
                val = results.get(m)
                row.append(round(float(val[i]), 8) if val is not None else "ERROR")
            writer.writerow(row)
    print(f"    Allocations --> {save_path}")


def save_diagnostics_csv(save_path, feature_names, utility, n_features):
    """
    Diagnostic utility values per feature:
      feature_name | v_singleton | v_N_minus_i | marginal_to_grand
    Plus summary rows for v_N and sum_singletons.
    """
    N   = list(range(n_features))
    v_N = utility(N)
    with open(save_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["metric", "value"])
        writer.writerow(["v_N (grand coalition accuracy)", round(v_N, 8)])
        writer.writerow(["sum_v_singletons", round(sum(utility([i]) for i in N), 8)])
        writer.writerow([])
        writer.writerow(["feature_name", "v_singleton", "v_N_minus_i", "marginal_to_grand"])
        for i, fname in enumerate(feature_names):
            v_s      = utility([i])
            v_loo    = utility([j for j in N if j != i])
            marginal = v_N - v_loo
            writer.writerow([fname, round(v_s, 8), round(v_loo, 8), round(marginal, 8)])
    print(f"    Diagnostics --> {save_path}")


# Main loop
for dataset_path in DATASETS:
    print(f"\n{'='*60}")
    print(f"DATASET: {dataset_path}")
    print(f"{'='*60}")

    data = np.load(dataset_path, allow_pickle=True)
    available_keys = list(data.keys())
    print(f"  Keys in archive: {available_keys}")

    required = ["X_train", "X_test", "y_train", "y_test"]
    if not all(k in available_keys for k in required):
        print(f"  [SKIP] Missing required keys. Expected {required}, found {available_keys}")
        continue

    X_train = data["X_train"]
    X_test  = data["X_test"]
    y_train = data["y_train"]
    y_test  = data["y_test"]
    feature_names = (data["feature_names"].tolist()
                     if "feature_names" in available_keys
                     else [f"f{i}" for i in range(X_train.shape[1])])

    n_features = X_train.shape[1]
    print(f"  X_train={X_train.shape}  X_test={X_test.shape}")
    print(f"  features={feature_names}")

    if n_features > MAX_FEATURES:
        print(f"  [SKIP] {n_features} features > MAX_FEATURES={MAX_FEATURES}")
        continue

    ds_name = os.path.basename(dataset_path).replace(".npz", "")

    # Per-dataset model override: house (n=16) skips random_forest (~2.7h)
    DATASET_MODEL_OVERRIDE = {
        "house_preprocessed": ["logistic", "svm"],
    }
    models_for_dataset = DATASET_MODEL_OVERRIDE.get(ds_name, MODELS)
    if models_for_dataset != MODELS:
        print(f"  [INFO] {ds_name}: using models={models_for_dataset} (override)")

    for model_name in models_for_dataset:
        print(f"\n  --- Model: {model_name} ---")

        utility = AccuracyUtility(X_train, X_test, y_train, y_test, model_name)
        u = UtilityWrapper(utility, feature_names)

        results = {}

        for method_name, compute_fn in [
            ("shapley",    lambda fn=exact_shapley_value: fn(u)),
            ("banzhaf",    lambda fn=exact_banzhaf_index: fn(u)),
            ("gately",     lambda fn=exact_gately_value:  fn(u)),
            ("least_core", lambda fn=exact_least_core:    fn(u)),
        ]:
            print(f"    Computing {method_name} ...", end=" ", flush=True)
            try:
                result = compute_fn()
                results[method_name] = result.values if hasattr(result, "values") else result
                print("done")
            except Exception as e:
                print(f"ERROR: {e}")
                results[method_name] = None

        alloc_path = os.path.join(OUTPUT_DIR, f"{ds_name}__{model_name}__allocations.csv")
        save_allocations_csv(alloc_path, feature_names, results, METHODS)

        diag_path = os.path.join(OUTPUT_DIR, f"{ds_name}__{model_name}__diagnostics.csv")
        save_diagnostics_csv(diag_path, feature_names, utility, n_features)

print("\nAll done.")


DATASET: ../datasets/house_preprocessed.npz
  Keys in archive: ['X_train', 'X_test', 'y_train', 'y_test', 'feature_names']
  X_train=(348, 16)  X_test=(87, 16)
  features=['handicapped-infants', 'water-project-cost-sharing', 'adoption-of-the-budget-resolution', 'physician-fee-freeze', 'el-salvador-aid', 'religious-groups-in-schools', 'anti-satellite-test-ban', 'aid-to-nicaraguan-contras', 'mx-missile', 'immigration', 'synfuels-corporation-cutback', 'education-spending', 'superfund-right-to-sue', 'crime', 'duty-free-exports', 'export-administration-act-south-africa']
  [INFO] house_preprocessed: using models=['logistic', 'svm'] (override)

  --- Model: logistic ---
    Computing shapley ... done
    Computing banzhaf ... done
    Computing gately ... done
    Computing least_core ... done
    Allocations --> ./output_final_allocations/house_preprocessed__logistic__allocations.csv
    Diagnostics --> ./output_final_allocations/house_preprocessed__logistic__diagnostics.csv

  --- Model: 

In [2]:
import numpy as np
from sklearn.metrics import accuracy_score
import sys
sys.path.insert(0, "")
from cgt_xai.models import get_model

data = np.load("../../datasets/bcw_preprocessed.npz", allow_pickle=True)
X_train, X_test = data["X_train"], data["X_test"]
y_train, y_test = data["y_train"], data["y_test"]

model = get_model("logistic", random_state=42)
model.fit(X_train[:, [0]], y_train)   # feature 0 = Clump_Thickness
print(accuracy_score(y_test, model.predict(X_test[:, [0]])))
# Should print ~0.857

0.8571428571428571
